# 02 - Total p-Variation (TpV) Reconstruction for Sparse-View CT

This notebook implements a model-based variational reconstruction method for sparse-view CT. The measured data are the degraded Mayo sinograms generated in notebook 01, and the reconstruction is obtained by solving the unconstrained optimization problem:

$$
\hat{x} = \arg\min_x \frac{1}{2}\|Kx-y^\delta\|_2^2 + \lambda \mathcal{R}_{TpV}(x)
$$

where:
- $K$ is the parallel-beam CT projector (Radon transform).
- $y^\delta$ is the noisy sinogram.
- $\mathcal{R}_{TpV}(x) = \sum_i \|\nabla x_i\|_2^p$ is the Total $p$-Variation prior, with $p$ in the non-convex sparse range $0.1 < p < 0.5$ as requested by the project specifications.

The solver is based on the **Chambolle-Pock Primal-Dual algorithm** (using `IPPy` tools) initialized from scratch (zeros) without any FBP initialization, to focus strictly on the variational paradigm. 

To comply with the project specifications (*"Students must ensure all methods use the same degraded inputs"*), **both this notebook and the UNet notebook evaluate on the exact same central test slice**, ensuring a perfectly fair and mathematically rigorous comparison between the variational and deep learning models.

## 1. Environment Setup and Imports

Mount the Google Drive, install `astra-toolbox` if needed, and import libraries from standard Python, PyTorch, and `IPPy`.

In [ ]:
!pip install astra-toolbox

from google.colab import drive
drive.mount("/content/drive")

import sys
from pathlib import Path
import json
import numpy as np
import torch
import matplotlib.pyplot as plt

# Paths setup
PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "tpv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities.metrics import PSNR, SSIM

# Set seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device configuration (CPU or GPU)
device = utilities.get_device()
print("Device used:", device)
print("Processed data directory:", PROCESSED_DIR)
print("TpV outputs directory:", OUTPUT_DIR)

## 2. Load the Preprocessed Data Contract (Test Patient)

Load the first test patient file to perform reconstruction on the exact same test slice used by downstream notebooks, ensuring consistency.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

# Load the first test patient to guarantee identical evaluation inputs across all methods
test_split = manifest["splits"]["test"]
patient_record = test_split["patients"][0]
patient_path = PROCESSED_DIR / patient_record["path"]
payload = torch.load(patient_path, map_location="cpu")

clean_images = payload["clean"].to(torch.float32)
sinograms = {key: val.to(torch.float32) for key, val in payload["sinograms"].items()}
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

print(f"Loaded Test Patient ID: {patient_id}")
print(f"Clean images batch shape: {tuple(clean_images.shape)}")
for views_key, sino in sinograms.items():
    print(f"Sinogram ({views_key} views) shape: {tuple(sino.shape)}")

## 3. Configure CT Projector and TpV Solver

Define the number of views to test (select from 180, 90, 60, 45) and initialize the parallel-beam CT projector $K$ and the unconstrained Chambolle-Pock TpV solver.

In [ ]:
# Set the angle setting to test (180, 90, 60, 45)
n_views = 60
angle_key = str(n_views)

if angle_key not in sinograms:
    raise ValueError(f"Angle config '{angle_key}' not found in preprocessed dataset.")

# Initialize physical parallel-beam projector
K = operators.CTProjector(
    img_shape=(256, 256),
    angles=np.linspace(0.0, np.pi, n_views),
    det_size=256,
    geometry="parallel",
    force_cpu=True,  # Set to False if CUDA/GPU support is fully configured in ASTRA
)

# Initialize the Chambolle-Pock TpV Solver
solver = solvers.ChambollePockTpVUnconstrained(K)
print(f"Initialized CT Projector and TpV Solver for {n_views} views.")

## 4. Hyperparameter Optimization on the Shared Test Slice

Run this cell to select the shared test slice, apply the Chambolle-Pock solver with your chosen $\lambda$ and $p$ hyperparameters, and compute the PSNR and SSIM. 

You can manually change the hyperparameters and rerun this cell multiple times to evaluate the performance on the same shared test slice.

In [ ]:
# -------------------- HYPERPARAMETERS TO TUNE --------------------
lmbda = 0.01          # Regularization parameter (weight of TpV penalty)
p = 0.35              # Sparsity parameter (strictly between 0.1 and 0.5)
maxiter = 150         # Number of maximum iterations
# -----------------------------------------------------------------

if not (0.1 < p < 0.5):
    raise ValueError(f"Project specifications require 0.1 < p < 0.5. Got p = {p}")

# Select the fixed central slice to guarantee identical evaluation inputs across all methods
slice_idx = clean_images.shape[0] // 2
print(f"Selected shared test slice index: {slice_idx}")

# Prepare 2D Ground Truth image and 2D corrupted sinogram on the selected device
x_true = clean_images[slice_idx : slice_idx + 1].to(device)
y_delta = sinograms[angle_key][slice_idx : slice_idx + 1].to(device)

print(f"Running Chambolle-Pock TpV solver (starting from scratch)... ")
start_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None
end_time = torch.cuda.Event(enable_timing=True) if torch.cuda.is_available() else None

if start_time:
    start_time.record()

# Execute the variational solver (starting_point=None starts from zeros)
x_sol, info = solver(
    y_delta,
    lmbda=lmbda,
    starting_point=None,  # Standard zero-image initialization
    x_true=x_true,
    maxiter=maxiter,
    tolf=1e-5,
    tolx=1e-5,
    p=p,
    verbose=True,
)

if end_time:
    end_time.record()
    torch.cuda.synchronize()
    print(f"Solved in {start_time.elapsed_time(end_time)/1000.0:.2f} seconds.")

# Normalize and clamp the final reconstructed image
x_sol = x_sol.detach().cpu()
x_true_cpu = x_true.detach().cpu()

x_sol_norm = normalize(x_sol).clamp(0.0, 1.0)

# Compute quantitative metrics according to project specifications (PSNR, SSIM)
psnr_val = PSNR(x_sol_norm, x_true_cpu)
ssim_val = SSIM(x_sol_norm, x_true_cpu)

print("\n--- QUANTITATIVE RESULTS ---")
print(f"PSNR: {psnr_val:.4f} dB")
print(f"SSIM: {ssim_val:.4f}")

## 5. Visual Inspection and Error Map

Plot the Ground Truth image, the reconstructed image under TpV, and the absolute error map $|TpV - GT|$ to inspect the spatial distribution of reconstruction errors. The generated visual comparison is saved in the `/outputs/tpv/` directory.

In [ ]:
# 1. Run FBP baseline solver for a fair side-by-side comparison
fbp_solver = solvers.FBP(K)
x_fbp, _ = fbp_solver(y_delta, x_true=None, starting_point=None)

def normalize_finite(image: torch.Tensor) -> torch.Tensor:
    image = torch.nan_to_num(image.float(), nan=0.0, posinf=0.0, neginf=0.0)
    if torch.isclose(image.max(), image.min()):
        return torch.zeros_like(image, dtype=torch.float32)
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)

x_fbp_norm = normalize_finite(x_fbp.detach().cpu())
fbp_psnr = float(PSNR(x_fbp_norm, x_true_cpu))
fbp_ssim = float(SSIM(x_fbp_norm, x_true_cpu))

# Convert to 2D numpy arrays for visualization
gt_np = x_true_cpu[0, 0].numpy()
fbp_np = x_fbp_norm[0, 0].numpy()
recon_np = x_sol_norm[0, 0].numpy()

# Compute error maps and crop slice for detail view
fbp_error = np.abs(fbp_np - gt_np)
tpv_error = np.abs(recon_np - gt_np)
crop_slice = (slice(96, 160), slice(96, 160))

# Plot 2x3 panel: GT, FBP, TpV and their respective center crop and error maps
fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
fig.suptitle(f"TpV Reconstruction & FBP Comparison - {n_views} views - Slice {slice_idx}\n(lmbda={lmbda}, p={p})", fontsize=14)

# First Row: Main images
axes[0, 0].imshow(gt_np, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 0].set_title("Ground Truth")
axes[0, 0].axis("off")

axes[0, 1].imshow(fbp_np, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 1].set_title(f"Noisy FBP Baseline\nPSNR: {fbp_psnr:.2f} dB | SSIM: {fbp_ssim:.4f}")
axes[0, 1].axis("off")

axes[0, 2].imshow(recon_np, cmap="gray", vmin=0.0, vmax=1.0)
axes[0, 2].set_title(f"TpV Reconstruction\nPSNR: {psnr_val:.2f} dB | SSIM: {ssim_val:.4f}")
axes[0, 2].axis("off")

# Second Row: Error maps and center crops
axes[1, 0].imshow(gt_np[crop_slice], cmap="gray", vmin=0.0, vmax=1.0)
axes[1, 0].set_title("GT Center Crop")
axes[1, 0].axis("off")

im_fbp_err = axes[1, 1].imshow(fbp_error, cmap="magma")
axes[1, 1].set_title("FBP Error |FBP - GT|")
axes[1, 1].axis("off")
fig.colorbar(im_fbp_err, ax=axes[1, 1], shrink=0.8)

im_tpv_err = axes[1, 2].imshow(tpv_error, cmap="magma")
axes[1, 2].set_title("TpV Error |TpV - GT|")
axes[1, 2].axis("off")
fig.colorbar(im_tpv_err, ax=axes[1, 2], shrink=0.8)

# Save visual comparison panel under /outputs/tpv/
output_fig_path = OUTPUT_DIR / f"tpv_comparison_{n_views}.png"
fig.savefig(output_fig_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved visual comparison panel:", output_fig_path)


## 6. Quantitative Results Comparison Table

Display a formatted comparison table of PSNR and SSIM metrics between the Filtered Backprojection (FBP) baseline and the Total p-Variation (TpV) reconstruction.

In [ ]:
# Print comparative metrics table
print("="*55)
print(f"{ 'METHOD':<20 } | { 'PSNR (dB)':<15 } | { 'SSIM':<12 }")
print("="*55)
print(f"{ 'FBP (Baseline)':<20 } | { fbp_psnr:<15.4f } | { fbp_ssim:<12.4f }")
print(f"{ 'TpV (Variational)':<20 } | { psnr_val:<15.4f } | { ssim_val:<12.4f }")
print("="*55)


## 7. Quantitative Comparison Plot and Export

Save the PSNR/SSIM comparison plot and a machine-readable metrics file for direct reuse in the final report and presentation.

In [ ]:
comparison_metrics = {
    "metadata": {
        "patient_id": patient_id,
        "slice_idx": int(slice_idx),
        "n_views": int(n_views),
        "lambda": float(lmbda),
        "p": float(p),
        "maxiter": int(maxiter),
    },
    "metrics": {
        "FBP (Baseline)": {"psnr": float(fbp_psnr), "ssim": float(fbp_ssim)},
        "TpV (Variational)": {"psnr": float(psnr_val), "ssim": float(ssim_val)},
    },
}

metrics_path = OUTPUT_DIR / f"tpv_metrics_{n_views}.json"
with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(comparison_metrics, file, indent=2)

methods = list(comparison_metrics["metrics"].keys())
psnr_values = [comparison_metrics["metrics"][method]["psnr"] for method in methods]
ssim_values = [comparison_metrics["metrics"][method]["ssim"] for method in methods]

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
fig.suptitle(f"Quantitative Comparison - {n_views} views - Slice {slice_idx}", fontsize=13)

axes[0].bar(methods, psnr_values, color=["tab:gray", "tab:blue"])
axes[0].set_ylabel("PSNR (dB)")
axes[0].set_title("PSNR")
axes[0].tick_params(axis="x", rotation=20)
for idx, value in enumerate(psnr_values):
    axes[0].text(idx, value, f"{value:.2f}", ha="center", va="bottom")

axes[1].bar(methods, ssim_values, color=["tab:gray", "tab:blue"])
axes[1].set_ylim(0.0, 1.0)
axes[1].set_ylabel("SSIM")
axes[1].set_title("SSIM")
axes[1].tick_params(axis="x", rotation=20)
for idx, value in enumerate(ssim_values):
    axes[1].text(idx, value, f"{value:.4f}", ha="center", va="bottom")

plot_path = OUTPUT_DIR / f"tpv_quantitative_comparison_{n_views}.png"
fig.savefig(plot_path, dpi=150)
plt.show()
plt.close(fig)

print("Saved quantitative metrics:", metrics_path)
print("Saved quantitative comparison plot:", plot_path)
